# Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim
from pyspark.sql.types import StringType
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
RENAME_MAP = {
    "ID": "id",
    "CAT": "category",
    "SUBCAT": "sub_category",
    "MAINTENANCE": "maintenance"
}

# Reading from Bronze layer

In [0]:
df = spark.table("data_lakehouse_project.bronze.erp_px_cat_g1v2")

# Data Transformation

- TRIM the strings
- Upper Column ID
- Rename Columns

In [0]:
df.display()

## Triming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## Upper ID column

In [0]:
df = df.withColumn("ID", F.upper(col("ID")))

In [0]:
df = df.withColumn("ID", F.regexp_replace(col("ID"), "_", "-"))

## Rename Columns

In [0]:
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

# Write into Silver Table

In [0]:
(
    df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("data_lakehouse_project.silver.erp_px_cat_g1v2")
)

# Check queries

In [0]:
%sql
SELECT *
FROM data_lakehouse_project.silver.erp_px_cat_g1v2

